# RAMSES ↔ pyCALIMA Equilibrium Comparison

This notebook loads a RAMSES G8 galaxy simulation (0.1 Z☉, `output_00081`) and compares the **time-evolved dust distribution** from the simulation against the **equilibrium dust distribution** predicted by pyCALIMA.

### Workflow
1. Load the RAMSES output with `yt`, define corrected dust fields
2. Explore the raw simulation data (T-nH phase space, σ, metallicity)
3. Build a 2D (T, nH) equilibrium grid using pyCALIMA's Newton-Krylov solver
4. Interpolate equilibrium predictions onto simulation cells
5. Compare: T-nH phase diagrams, DTM distributions, mass by size/composition

### Key corrections
- **Silicate SioverSil = 0.163**: `dust_bin03` and `dust_bin04` in the RAMSES output store the *Si mass fraction*, not the total silicate mass. Actual silicate density = stored × ρ_gas / 0.163 (Dubois+ 2024 convention).
- **G0 = 1 Habing**: The UV field is not stored in the RAMSES output; G0 = 1 is assumed throughout.
- **σ_turb from `scalar_14`**: The 20th hydro variable is the local turbulent velocity dispersion (km/s).

In [ ]:
from pycalima._paths import get_results_dir, resolve_solver_config_path
%matplotlib inline
import sys, os, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import yt
from scipy.interpolate import RegularGridInterpolator

# ── pyCALIMA path ──────────────────────────────────────────────────────────────

from pycalima.solvers.run_grid import run_grid, save_grid_npz, load_grid_npz

yt.set_log_level('critical')  # suppress yt verbose output
print('yt version:', yt.__version__)

In [ ]:
# ── Paths and physical constants ───────────────────────────────────────────────
RAMSES_OUTPUT = os.path.expanduser(
    '~/Documents/RAMSES_dev/pyCALIMA_testing/test_outputs'
    '/CorrectSN_0.1Zsun_DTMini1d-3_MR/output_00081'
)
CONFIG_PATH   = str(resolve_solver_config_path('ramses_G8_0.1Zsun'))
GRID_CACHE    = str(get_results_dir() / 'eq_grid_G8_0.1Zsun.npz')

SioverSil = 0.163    # Si mass fraction inside silicate grain (Dubois+ 2024)
G0_ASSUMED = 1.0     # Habing units — not stored in RAMSES output
ZSUN  = 0.0134       # Solar metallicity, Asplund+ 2009
MH    = 1.6726e-24   # proton mass [g]
MU    = 1.4          # mean molecular weight (neutral ISM default)
KB    = 1.3806e-16   # Boltzmann constant [erg/K]

BOX_HALF_KPC = 4.0   # restrict analysis to a cube of side 2×4 = 8 kpc

# ── Working directory and results folder ──────────────────────────────────────
os.makedirs('results', exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'Analysis box: {2*BOX_HALF_KPC:.0f} kpc cube centred on domain centre')

## 1. Load RAMSES output and define derived fields

The simulation has 20 hydro variables (see `hydro_file_descriptor.txt`): density, 3 velocities, pressure, metallicity (gas-phase), 8 element mass fractions (H,O,Fe,Mg,C,N,Si,S), D, 4 dust bins, and `scalar_14` (σ_turb).

> **Note**: `eq_analysis.py` in `models/tools/` uses field names from an *older* RAMSES simulation (`CSmall`, `SilSmall`, etc.). Here we use the field names as written in `hydro_file_descriptor.txt` of this specific output.

In [ ]:
def register_pyCALIMA_fields(ds, SioverSil=0.163):
    """Register corrected dust fields for this RAMSES output.

    yt 4.x naming for this RAMSES output (from hydro_file_descriptor.txt):
        ('ramses', 'Density')            — gas mass density  [capitalised]
        ('ramses', 'Metallicity')        — gas-phase metal mass fraction
        ('ramses', 'hydro_dust_bin01')   — small carb dust mass frac
        ('ramses', 'hydro_dust_bin02')   — large carb dust mass frac
        ('ramses', 'hydro_dust_bin03')   — small sil Si-fraction
        ('ramses', 'hydro_dust_bin04')   — large sil Si-fraction
        ('ramses', 'hydro_scalar_14')    — σ_turb [km/s, stored as dimensionless float]
        ('ramses', 'hydro_chem_C/Si/...')— element mass fracs

    ('gas', 'density') and ('gas', 'temperature') are yt-derived from the above.
    """
    _Si = SioverSil

    def _nH(field, data):
        return (data[('gas', 'density')] / (MU * yt.units.mh)).to('1/cm**3')
    ds.add_field(('gas', 'nH'), function=_nH, units='1/cm**3',
                 sampling_type='cell', force_override=True)

    def _sigma(field, data):
        # scalar_14 stores σ_turb in physical km/s as a dimensionless float;
        # attach km/s units directly rather than calling .to() on a dimensionless array
        return data[('ramses', 'hydro_scalar_14')] * yt.units.km / yt.units.s
    ds.add_field(('gas', 'sigma_turb'), function=_sigma, units='km/s',
                 sampling_type='cell', force_override=True)

    def _cs(field, data):
        return (data[('ramses', 'hydro_dust_bin01')] * data[('gas', 'density')]).to('g/cm**3')
    ds.add_field(('gas', 'rho_carb_small'), function=_cs, units='g/cm**3',
                 sampling_type='cell', force_override=True)

    def _cl(field, data):
        return (data[('ramses', 'hydro_dust_bin02')] * data[('gas', 'density')]).to('g/cm**3')
    ds.add_field(('gas', 'rho_carb_large'), function=_cl, units='g/cm**3',
                 sampling_type='cell', force_override=True)

    def _ss(field, data):
        return (data[('ramses', 'hydro_dust_bin03')] * data[('gas', 'density')] / _Si).to('g/cm**3')
    ds.add_field(('gas', 'rho_sil_small'), function=_ss, units='g/cm**3',
                 sampling_type='cell', force_override=True)

    def _sl(field, data):
        return (data[('ramses', 'hydro_dust_bin04')] * data[('gas', 'density')] / _Si).to('g/cm**3')
    ds.add_field(('gas', 'rho_sil_large'), function=_sl, units='g/cm**3',
                 sampling_type='cell', force_override=True)


ds = yt.load(RAMSES_OUTPUT)
register_pyCALIMA_fields(ds)

print(f'Simulation time: {ds.current_time.to("Myr"):.1f}')
print(f'Box size:        {ds.domain_width[0].to("kpc"):.0f}')
print(f'Max AMR level:   {ds.max_level}')
print(f'Available ramses fields: {[f[1] for f in ds.field_list if f[0]=="ramses"]}')

## 2. Sanity-check projections

Three yt projection plots to verify the dataset loaded correctly: column gas density, mass-weighted temperature, and mass-weighted gas-phase metallicity, all projected along the z-axis.

In [ ]:
os.makedirs('results', exist_ok=True)

PROJ_AXIS  = 'z'
PROJ_WIDTH = (2 * BOX_HALF_KPC, 'kpc')   # 8 kpc window — same as analysis box

PROJ_FIELDS = [
    # (field,                      weight_field,          output_name,  cmap)
    (('gas', 'density'),           None,                  'density',    'magma'),
    (('gas', 'temperature'),       ('gas', 'density'),    'temperature','RdYlBu_r'),
    (('ramses', 'Metallicity'),    ('gas', 'density'),    'metallicity','viridis'),
]

for field, weight, name, cmap in PROJ_FIELDS:
    p = yt.ProjectionPlot(ds, PROJ_AXIS, field, weight_field=weight,
                          center=ds.domain_center, width=PROJ_WIDTH)
    p.set_cmap(field, cmap)
    p.set_font({'size': 14})
    fname = f'results/proj_{name}.png'
    p.save(fname)
    print(f'Saved {fname}')
    p.show()

## 3. Extract cell data as numpy arrays

In [ ]:
# ── Restrict to 8 kpc cube centred on the domain centre ──────────────────────
half = ds.quan(BOX_HALF_KPC, 'kpc')
ad   = ds.box(ds.domain_center - half, ds.domain_center + half)
print(f'Cells in 8 kpc box: {ad[("gas","density")].shape[0]:,}')

# ── Gas state ─────────────────────────────────────────────────────────────────
rho   = ad[('gas', 'density')].to('g/cm**3').value          # g/cm³
T_gas = ad[('gas', 'temperature')].to('K').value             # K
nH    = ad[('gas', 'nH')].to('1/cm**3').value               # cm⁻³
sigma = ad[('gas', 'sigma_turb')].to('km/s').value           # km/s
vol   = ad[('index', 'cell_volume')].to('cm**3').value       # cm³

# ── Metallicity (point 3): 'Metallicity' = total metal mass fraction (gas+dust)
#    No approximation needed — use the RAMSES field directly.
Z_metal = ad[('ramses', 'Metallicity')].value                # gas+dust metals/ρ
rho_metal = Z_metal * rho                                    # total metal mass density [g/cm³]

# ── Individual element mass fractions (point 4): chem_X = gas+dust combined ──
fH  = ad[('ramses', 'hydro_chem_H')].value
fC  = ad[('ramses', 'hydro_chem_C')].value
fN  = ad[('ramses', 'hydro_chem_N')].value
fO  = ad[('ramses', 'hydro_chem_O')].value
fMg = ad[('ramses', 'hydro_chem_Mg')].value
fSi = ad[('ramses', 'hydro_chem_Si')].value
fS  = ad[('ramses', 'hydro_chem_S')].value
fFe = ad[('ramses', 'hydro_chem_Fe')].value

# ── Corrected absolute dust mass densities [g/cm³] ───────────────────────────
rho_cs = ad[('gas', 'rho_carb_small')].to('g/cm**3').value
rho_cl = ad[('gas', 'rho_carb_large')].to('g/cm**3').value
rho_ss = ad[('gas', 'rho_sil_small')].to('g/cm**3').value
rho_sl = ad[('gas', 'rho_sil_large')].to('g/cm**3').value
rho_dust = rho_cs + rho_cl + rho_ss + rho_sl

# ── Cell mass for mass-weighting ──────────────────────────────────────────────
cell_mass = rho * vol

# ── DTM: dust mass / total metal mass (both gas+dust metals in denominator) ──
with np.errstate(invalid='ignore', divide='ignore'):
    DTM_sim = np.where(rho_metal > 0, rho_dust / rho_metal, 0.0)

print(f'Total gas mass:  {cell_mass.sum():.3e} g')
print(f'Total dust mass: {(rho_dust * vol).sum():.3e} g')
print(f'Global DTM:      {(rho_dust * vol).sum() / (rho_metal * vol).sum():.4f}')
print(f'Mean Z_metal:    {np.average(Z_metal, weights=cell_mass):.4e}  '
      f'({np.average(Z_metal, weights=cell_mass)/ZSUN:.3f} Zsun)')

## 4. Sanity-check distributions

Mass-weighted histograms of the three key ISM state variables, plus the T-nH phase diagram.  These four panels confirm the simulation spans the expected temperature, density, and metallicity ranges.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

mask = (T_gas > 0) & (nH > 0) & (Z_metal > 0)
mw   = cell_mass[mask]

# ── Temperature histogram ─────────────────────────────────────────────────────
ax = axes[0]
ax.hist(np.log10(T_gas[mask]), bins=120, weights=mw, density=True,
        color='steelblue', alpha=0.85)
T_mw = np.average(np.log10(T_gas[mask]), weights=mw)
ax.axvline(T_mw, color='red', lw=2, label=f'Mean = {10**T_mw:.0f} K')
ax.set_xlabel(r'$\log_{10}(T\ [\mathrm{K}])$')
ax.set_ylabel('Mass-weighted PDF')
ax.set_title('Temperature')
ax.legend(fontsize=9)

# ── nH histogram ─────────────────────────────────────────────────────────────
ax = axes[1]
ax.hist(np.log10(nH[mask]), bins=120, weights=mw, density=True,
        color='darkorange', alpha=0.85)
nH_mw = np.average(np.log10(nH[mask]), weights=mw)
ax.axvline(nH_mw, color='red', lw=2,
           label=fr'Mean = {10**nH_mw:.2g} cm$^{{-3}}$')
ax.set_xlabel(r'$\log_{10}(n_H\ [\mathrm{cm}^{-3}])$')
ax.set_xlim(-4, 3)
ax.set_title('Hydrogen number density')
ax.legend(fontsize=9)

# ── Metallicity histogram — use total metal fraction directly ─────────────────
ax = axes[2]
ax.hist(np.log10(Z_metal[mask] / ZSUN), bins=120, weights=mw, density=True,
        color='forestgreen', alpha=0.85)
Z_mw = np.average(np.log10(Z_metal[mask] / ZSUN), weights=mw)
ax.axvline(Z_mw, color='red', lw=2, label=f'Mean = {10**Z_mw:.3f} Z☉')
ax.axvline(np.log10(0.1), color='gray', lw=1.5, ls='--', label='0.1 Z☉ (IC)')
ax.set_xlabel(r'$\log_{10}(Z_\mathrm{total}/Z_\odot)$')
ax.set_title('Total metallicity (gas+dust)')
ax.legend(fontsize=9)

# ── T-nH phase diagram ───────────────────────────────────────────────────────
ax = axes[3]
h, xe, ye = np.histogram2d(
    np.log10(nH[mask]), np.log10(T_gas[mask]),
    bins=120, range=[(-4, 3), (1, 8)], weights=mw
)
im = ax.pcolormesh(xe, ye, np.log10(h.T + 1e-50), cmap='viridis')
plt.colorbar(im, ax=ax, label=r'$\log_{10}$(mass) [g]')
ax.set_xlabel(r'$\log_{10}(n_H\ [\mathrm{cm}^{-3}])$')
ax.set_ylabel(r'$\log_{10}(T\ [\mathrm{K}])$')
ax.set_title('T-nH phase diagram (mass-weighted)')

plt.suptitle(
    f'Sanity check — 8 kpc box: {len(T_gas):,} cells, '
    fr'$M_\mathrm{{gas}} = {cell_mass.sum():.2e}$ g',
    fontsize=11
)
plt.tight_layout()
plt.savefig('results/sanity_check_distributions.png', dpi=150, bbox_inches='tight')


## 5. Detailed exploration: phase space, σ_turb, and metallicity budget

Before building the equilibrium grid we need to know the T-nH range covered by the simulation and the typical turbulent velocity dispersion (which sets the turbulence parameter in the config).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

mask = (T_gas > 0) & (nH > 0)

# ── Panel 1: mass-weighted T-nH phase diagram ─────────────────────────────────
ax = axes[0]
h, xe, ye = np.histogram2d(
    np.log10(nH[mask]), np.log10(T_gas[mask]),
    bins=100, range=[(-4, 3), (1, 8)],
    weights=cell_mass[mask]
)
im0 = ax.pcolormesh(xe, ye, np.log10(h.T + 1e-50), cmap='viridis', vmin=None)
plt.colorbar(im0, ax=ax, label=r'$\log_{10}$(mass) [g]')
ax.set_xlabel(r'$\log_{10}(n_H\ [\mathrm{cm}^{-3}])$')
ax.set_ylabel(r'$\log_{10}(T\ [\mathrm{K}])$')
ax.set_title('T-nH phase diagram (mass-weighted)')

# ── Panel 2: mass-weighted sigma distribution ─────────────────────────────────
ax = axes[1]
smask = (sigma > 0) & mask
ax.hist(sigma[smask], bins=100, weights=cell_mass[smask], density=True, log=True,
        color='steelblue', alpha=0.8)
sigma_mw = np.average(sigma[smask], weights=cell_mass[smask])
sigma_p25, sigma_p75 = np.percentile(sigma[smask], [25, 75])
ax.axvline(sigma_mw, color='red',    lw=2, label=f'Mean = {sigma_mw:.2f} km/s')
ax.axvline(sigma_p25, color='orange', lw=1.5, ls='--', label=f'p25 = {sigma_p25:.2f} km/s')
ax.axvline(sigma_p75, color='orange', lw=1.5, ls='-.', label=f'p75 = {sigma_p75:.2f} km/s')
ax.set_xlabel(r'$\sigma_\mathrm{turb}$ [km/s]')
ax.set_title('Turbulent velocity dispersion (mass-wtd)')
ax.legend(fontsize=9)

# ── Panel 3: total metallicity distribution (from Metallicity field directly) ─
ax = axes[2]
Zmask = mask & (Z_metal > 0)
ax.hist(np.log10(Z_metal[Zmask] / ZSUN), bins=100,
        weights=cell_mass[Zmask], density=True,
        color='darkorange', alpha=0.8)
Z_mw_log = np.average(np.log10(Z_metal[Zmask] / ZSUN), weights=cell_mass[Zmask])
ax.axvline(Z_mw_log, color='red', lw=2, label=f'Mean = {10**Z_mw_log:.3f} Z☉')
ax.axvline(np.log10(0.1), color='gray', lw=1.5, ls='--', label='0.1 Z☉ (IC)')
ax.set_xlabel(r'$\log_{10}(Z_\mathrm{total}/Z_\odot)$')
ax.set_title('Total metallicity (gas+dust, Metallicity field)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('results/exploration_phase_sigma_Z.png', dpi=150, bbox_inches='tight')

# ── Grid design stats ─────────────────────────────────────────────────────────
T_p05, T_p95   = np.percentile(T_gas[mask], [5, 95])
nH_p05, nH_p95 = np.percentile(nH[mask], [5, 95])

print(f'T  5th–95th: {T_p05:.1e} – {T_p95:.1e} K')
print(f'nH 5th–95th: {nH_p05:.2e} – {nH_p95:.2e} cm⁻³')
print(f'sigma mass-weighted mean: {sigma_mw:.2f} km/s')

## 6. Simulation dust distribution on T-nH plane

In [ ]:
BIN_LABELS  = ['Carb small\n(dust_bin01)', 'Carb large\n(dust_bin02)',
               'Sil small\n(dust_bin03×1/SioverSil)', 'Sil large\n(dust_bin04×1/SioverSil)']
SIM_BINS    = [rho_cs, rho_cl, rho_ss, rho_sl]

NBINS_2D = 80
NH_RANGE = (-4, 3)   # nH: 1e-4 to 1e3 cm-3
T_RANGE  = (1, 8)


def phase_rho_dust(T, nH, rho_bins, labels, vol_weights, figsize=(20, 4.5),
                   vmin=-35, vmax=-27, title=''):
    """2D histogram of T-nH coloured by volume-weighted mean dust mass density."""
    fig, axes = plt.subplots(1, len(rho_bins), figsize=figsize, sharey=True)
    log_nH = np.log10(np.clip(nH, 1e-5, 1e4))
    log_T  = np.log10(np.clip(T,  1e0,  1e9))

    for ax, rho_bin, label in zip(axes, rho_bins, labels):
        h_w, xe, ye = np.histogram2d(log_nH, log_T, bins=NBINS_2D,
                                      range=[NH_RANGE, T_RANGE],
                                      weights=rho_bin * vol_weights)
        h_v, _,  _  = np.histogram2d(log_nH, log_T, bins=NBINS_2D,
                                      range=[NH_RANGE, T_RANGE],
                                      weights=vol_weights)
        with np.errstate(invalid='ignore', divide='ignore'):
            mean_rho = np.where(h_v > 0, h_w / h_v, np.nan)
        im = ax.pcolormesh(xe, ye, np.log10(np.abs(mean_rho.T) + 1e-50),
                           cmap='inferno', vmin=vmin, vmax=vmax)
        ax.set_xlabel(r'$\log_{10}(n_H\ [\mathrm{cm}^{-3}])$')
        ax.set_title(label, fontsize=10)
        plt.colorbar(im, ax=ax, label=r'$\log_{10}(\rho_\mathrm{dust})$ [g/cm³]')

    axes[0].set_ylabel(r'$\log_{10}(T\ [\mathrm{K}])$')
    if title:
        fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    return fig


fig_sim = phase_rho_dust(
    T_gas, nH, SIM_BINS, BIN_LABELS, vol,
    title='RAMSES simulation: dust mass density on T-nH plane'
)
plt.savefig('results/sim_dust_Tnplane.png', dpi=150, bbox_inches='tight')


## 7. Build pyCALIMA equilibrium grid

We use `run_grid()` from `solvers/run_grid.py` to compute the Newton-Krylov equilibrium over a 2D (T, nH) grid, then save to `.npz` for reuse.

The config `ramses_G8_0.1Zsun.json` is updated with the mass-weighted median σ from the simulation before running.

In [ ]:
# ── Rémy-Ruyer+2014 gas-to-dust ratio ────────────────────────────────────────
def remy_ruyer_dtm(Z_Zsun):
    """DTM from Rémy-Ruyer et al. (2014) single power-law fit.

    log(GDR) = 2.21 – 2.21 × log(Z/Z_sun)
    GDR_sun = 10^2.21 ≈ 162, giving DTM_sun ≈ 0.46.
    """
    GDR = 10.0**2.21 * Z_Zsun**(-2.21)
    Z_actual = Z_Zsun * ZSUN
    return 1.0 / (GDR * Z_actual)          # DTM = (1/GDR) / Z_mass_fraction

# ── Zubko+2004 BARE-GR-S composition fractions ───────────────────────────────
# From the docstring in dust_oppacity.py (compute_zubko2004_bare_gr_s_cross_sections):
#   Total dust mass fractions: PAH 4.57%, graphite 29.47%, silicate 65.96%
# We have no PAH bins — normalise carbonaceous + silicate to 100%.
_f_carb_zubko = 0.2947
_f_sil_zubko  = 0.6596
_f_tot         = _f_carb_zubko + _f_sil_zubko
F_CARB = _f_carb_zubko / _f_tot    # 0.309 — fraction of dust in carbonaceous
F_SIL  = _f_sil_zubko  / _f_tot    # 0.691 — fraction of dust in silicate

# Small/large mass split within each composition: integrate the Zubko dn/da
# over the RAMSES bin size ranges to get the fraction in each lognormal bin.
import sys as _sys
from pycalima.models.dust_radiation.dust_oppacity import (
    _zubko_dnda_graphite, _zubko_dnda_silicate
)

# Avoid np.trapz (removed in NumPy 2.0); use lazy getattr so no AttributeError
_trapz = getattr(np, 'trapezoid', None) or getattr(np, 'trapz', None)

_a = np.logspace(np.log10(3.5e-4), np.log10(0.37), 4000)  # µm

def _mass_fraction_in_range(dnda_func, rho_g_cm3, a_grid, alo, ahi):
    """Fraction of total Zubko mass in the size range [alo, ahi] µm."""
    mask = (a_grid >= alo) & (a_grid <= ahi)
    dm = (4.0/3.0) * np.pi * rho_g_cm3 * (a_grid * 1e-4)**3 * dnda_func(a_grid)
    m_range = _trapz(dm[mask], a_grid[mask])
    m_total = _trapz(dm, a_grid)
    return m_range / m_total if m_total > 0 else 0.0

# Graphite (rho=2.24 g/cm³): small=[0.001,0.030] µm, large=[0.030,0.300] µm
_fs_carb = _mass_fraction_in_range(_zubko_dnda_graphite, 2.24, _a, 0.001, 0.030)
_fl_carb = 1.0 - _fs_carb
# Silicate (rho=3.30 g/cm³): small=[0.002,0.015] µm, large=[0.015,0.300] µm
_fs_sil  = _mass_fraction_in_range(_zubko_dnda_silicate, 3.30, _a, 0.002, 0.015)
_fl_sil  = 1.0 - _fs_sil

# Per-bin weight: [small_carb, large_carb, small_sil, large_sil]
BIN_WEIGHTS = np.array([F_CARB*_fs_carb, F_CARB*_fl_carb,
                         F_SIL*_fs_sil,   F_SIL*_fl_sil])
print(f'Zubko 2004 composition split (no PAH):')
print(f'  Carbonaceous: {F_CARB*100:.1f}%  (small {F_CARB*_fs_carb*100:.2f}%,'
      f' large {F_CARB*_fl_carb*100:.2f}%)')
print(f'  Silicate:     {F_SIL*100:.1f}%  (small {F_SIL*_fs_sil*100:.2f}%,'
      f' large {F_SIL*_fl_sil*100:.2f}%)')

# ── Solar metal mass fractions (Asplund+2009, Z_sun=0.0134) ──────────────────
SOLAR_METAL_MF = {
    'C':  2.369e-3,
    'N':  8.320e-4,
    'O':  5.735e-3,
    'Mg': 6.440e-4,
    'Si': 6.710e-4,
    'S':  3.410e-4,
    'Fe': 1.258e-3,
}
HE_SOLAR = 0.2485           # He mass fraction (kept ~constant)

# ── 4D grid axes ─────────────────────────────────────────────────────────────
T_p05, T_p95   = np.percentile(T_gas[(T_gas > 0) & (nH > 0)], [5, 95])
nH_p05, nH_p95 = np.percentile(nH[(T_gas > 0) & (nH > 0)],   [5, 95])

T_grid  = np.logspace(max(1.0, np.floor(np.log10(T_p05))),
                       min(8.0, np.ceil(np.log10(T_p95))), 15)
nH_grid = np.logspace(max(-4.0, np.floor(np.log10(nH_p05))),
                       min(3.0,  np.ceil(np.log10(nH_p95))), 15)

Z_grid_Zsun = np.array([0.01, 0.033, 0.1, 0.33, 1.0])   # Z / Z_sun
sigma_grid  = np.array([0.05, 0.2, 0.8, 2.5, 8.0])       # km/s

print(f'\n4D equilibrium grid:')
print(f'  T:     {T_grid[0]:.1e} – {T_grid[-1]:.1e} K  ({len(T_grid)} pts)')
print(f'  nH:    {nH_grid[0]:.1e} – {nH_grid[-1]:.1e} cm⁻³ ({len(nH_grid)} pts)')
print(f'  Z:     {Z_grid_Zsun} Z_sun')
print(f'  sigma: {sigma_grid} km/s')
print(f'  Total:  {len(T_grid)*len(nH_grid)*len(Z_grid_Zsun)*len(sigma_grid):,} NK solves')
print(f'  ~Est:  {len(T_grid)*len(nH_grid)*len(Z_grid_Zsun)*len(sigma_grid)*5/8/60:.0f} min with 8 workers @ 5 s/pt')

In [ ]:
from joblib import Parallel, delayed

N_SPOT   = 100          # number of cells to test
NH_FLOOR = 10.0         # cm⁻³ — "high-density gas"

# ── Select N_SPOT cells from the dense ISM, mass-weighted ────────────────────
dense_mask = (nH > NH_FLOOR) & (T_gas > 0) & (Z_metal > 0) & (sigma > 0)
print(f'Dense cells (nH > {NH_FLOOR} cm⁻³): {dense_mask.sum():,} '
      f'({dense_mask.sum()/len(nH)*100:.1f}% of all cells)')

rng = np.random.default_rng(42)
dense_idx = np.where(dense_mask)[0]
w = cell_mass[dense_idx]
chosen = rng.choice(dense_idx, size=min(N_SPOT, len(dense_idx)),
                    replace=False, p=w/w.sum())
print(f'Selected {len(chosen)} cells  '
      f'(nH range: {nH[chosen].min():.1f}–{nH[chosen].max():.0f} cm⁻³, '
      f'T range: {T_gas[chosen].min():.0f}–{T_gas[chosen].max():.0e} K)')

# ── Per-cell NK solve ─────────────────────────────────────────────────────────
nH_ref_cfg = json.load(open(CONFIG_PATH))['environment']['hydrogen_number_density_cm3']

def _solve_cell(i, cell_idx):
    """Run the NK equilibrium solver for one simulation cell."""
    T_c   = float(T_gas[cell_idx])
    nH_c  = float(nH[cell_idx])
    Z_c   = float(Z_metal[cell_idx])        # total metal mass fraction
    sig_c = float(sigma[cell_idx])          # km/s

    cfg = _build_cfg_for_Zsig(CONFIG_PATH,
                               Z_Zsun=Z_c / ZSUN,
                               sigma_km_s=sig_c,
                               nH_ref_cfg=nH_ref_cfg)
    tmp = f'results/_spot_{i:03d}.json'
    with open(tmp, 'w') as f:
        json.dump(cfg, f)

    grid = run_grid(tmp,
                    x_param='T',  x_values=[T_c],
                    y_param='nH', y_values=[nH_c],
                    solver_type='newton_krylov',
                    n_jobs=1, verbose=False)
    try:
        os.remove(tmp)
    except OSError:
        pass

    rho_dust_eq = grid['rho_dust'][0, 0, :]          # g/cm³, shape (4,)
    conv        = bool(grid['converged'][0, 0])

    # DTM_eq: dust mass / metal mass at this cell's Z and nH
    rho_metal_c = Z_c * nH_c * MH * MU               # ρ_metal from eq. config Z
    dtm_eq      = rho_dust_eq.sum() / rho_metal_c if rho_metal_c > 0 else np.nan

    # DTM_sim for this cell
    dtm_sim     = float(DTM_sim[cell_idx])

    return dict(dtm_eq=dtm_eq, dtm_sim=dtm_sim, converged=conv,
                T=T_c, nH=nH_c, Z=Z_c, sigma=sig_c,
                rho_dust_eq=rho_dust_eq)

import time as _time
t0 = _time.perf_counter()
results = Parallel(n_jobs=8, verbose=5)(
    delayed(_solve_cell)(i, idx) for i, idx in enumerate(chosen)
)
print(f'\nDone in {(_time.perf_counter()-t0)/60:.1f} min')

# ── Unpack ────────────────────────────────────────────────────────────────────
dtm_eq_spot  = np.array([r['dtm_eq']    for r in results])
dtm_sim_spot = np.array([r['dtm_sim']   for r in results])
conv_spot    = np.array([r['converged'] for r in results])
T_spot       = np.array([r['T']         for r in results])
nH_spot      = np.array([r['nH']        for r in results])
Z_spot       = np.array([r['Z']         for r in results])
sig_spot     = np.array([r['sigma']     for r in results])

print(f'Convergence: {conv_spot.sum()}/{len(conv_spot)} cells')
print(f'DTM sim  — median: {np.nanmedian(dtm_sim_spot):.3f}  '
      f'range: {np.nanmin(dtm_sim_spot):.3f}–{np.nanmax(dtm_sim_spot):.3f}')
print(f'DTM eq   — median: {np.nanmedian(dtm_eq_spot):.3f}  '
      f'range: {np.nanmin(dtm_eq_spot[conv_spot]):.3f}–{np.nanmax(dtm_eq_spot[conv_spot]):.3f}')

# ── Scatter plot: sim vs eq ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: 1-to-1 scatter coloured by T
ax = axes[0]
sc = ax.scatter(
    np.log10(np.clip(dtm_eq_spot[conv_spot],  1e-6, 1)),
    np.log10(np.clip(dtm_sim_spot[conv_spot], 1e-6, 1)),
    c=np.log10(T_spot[conv_spot]), cmap='RdYlBu_r', vmin=1, vmax=7,
    s=60, edgecolors='k', linewidths=0.4, zorder=3
)
lims = (-5, 0.5)
ax.plot(lims, lims, 'k--', lw=1.5, label='1:1')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel(r'$\log_{10}$(DTM equilibrium)')
ax.set_ylabel(r'$\log_{10}$(DTM simulation)')
ax.set_title(f'Spot check: {conv_spot.sum()} converged cells  (nH > {NH_FLOOR} cm⁻³)')
plt.colorbar(sc, ax=ax, label=r'$\log_{10}(T\ [\mathrm{K}])$')
ax.legend()

# Panel 2: ratio vs nH coloured by T
ax = axes[1]
ratio = np.log10(np.clip(dtm_sim_spot[conv_spot], 1e-8, None)
               / np.clip(dtm_eq_spot[conv_spot],  1e-8, None))
sc2 = ax.scatter(np.log10(nH_spot[conv_spot]), ratio,
                 c=np.log10(T_spot[conv_spot]), cmap='RdYlBu_r', vmin=1, vmax=7,
                 s=60, edgecolors='k', linewidths=0.4, zorder=3)
ax.axhline(0, color='k', lw=1.5, ls='--')
ax.axhline(+1, color='gray', lw=1, ls=':')
ax.axhline(-1, color='gray', lw=1, ls=':')
ax.set_xlabel(r'$\log_{10}(n_H\ [\mathrm{cm}^{-3}])$')
ax.set_ylabel(r'$\log_{10}(\mathrm{DTM_{sim}} / \mathrm{DTM_{eq}})$')
ax.set_title('Ratio sim/eq vs density')
plt.colorbar(sc2, ax=ax, label=r'$\log_{10}(T\ [\mathrm{K}])$')

plt.tight_layout()
plt.savefig('results/spot_check_DTM.png', dpi=150, bbox_inches='tight')


## 8. Inspect equilibrium grid convergence and structure

In [ ]:
# ── Find the grid slice nearest the simulation's mean Z and sigma ─────────────
Z_metal_mw   = np.average(Z_metal, weights=cell_mass)
sigma_mw     = np.average(sigma[(sigma > 0)], weights=cell_mass[(sigma > 0)])

i_Z_slice   = np.argmin(np.abs(Z_grid_Zsun - Z_metal_mw / ZSUN))
i_sig_slice = np.argmin(np.abs(sigma_grid - sigma_mw))

Z_slice  = Z_grid_Zsun[i_Z_slice]
sig_slice = sigma_grid[i_sig_slice]
print(f'Sim mean Z = {Z_metal_mw/ZSUN:.3f} Z☉  → grid slice at {Z_slice:.3f} Z☉')
print(f'Sim mean σ = {sigma_mw:.3f} km/s → grid slice at {sig_slice:.3f} km/s')

# ── DTM at the chosen (Z, sigma) slice ────────────────────────────────────────
rho_gas_grid   = nH_grid[np.newaxis, :] * MH * MU          # (1, nNH) g/cm³
rho_metal_slice = Z_slice * ZSUN * rho_gas_grid             # (1, nNH)
rho_dust_slice  = rho_dust_4d[:, :, i_Z_slice, i_sig_slice, :]  # (nT, nNH, 4)
DTM_slice = rho_dust_slice / rho_metal_slice[:, :, np.newaxis]   # (nT, nNH, 4)

print(f'\nConvergence at this slice: '
      f'{converged_4d[:,:,i_Z_slice,i_sig_slice].mean()*100:.1f}%')

# ── Plot DTM on T-nH plane for each bin ──────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), sharey=True)
log_T_g  = np.log10(T_grid)
log_nH_g = np.log10(nH_grid)

for k, (ax, label) in enumerate(zip(axes, BIN_LABELS)):
    vals = np.log10(np.clip(DTM_slice[:, :, k], 1e-6, 1.0))
    im = ax.pcolormesh(log_nH_g, log_T_g, vals, cmap='plasma', vmin=-5, vmax=0)
    nc = ~converged_4d[:, :, i_Z_slice, i_sig_slice].astype(bool)
    if nc.any():
        nH_nc = log_nH_g[np.where(nc)[1]]
        T_nc  = log_T_g[np.where(nc)[0]]
        ax.scatter(nH_nc, T_nc, s=5, c='white', marker='x', label='Not converged')
    ax.set_xlabel(r'$\log_{10}(n_H\ [\mathrm{cm}^{-3}])$')
    ax.set_title(label, fontsize=10)
    plt.colorbar(im, ax=ax, label=r'$\log_{10}$(DTM)')

axes[0].set_ylabel(r'$\log_{10}(T\ [\mathrm{K}])$')
fig.suptitle(
    f'pyCALIMA equilibrium DTM  (G0={G0_ASSUMED}, Z={Z_slice:.3f} Z☉, '
    f'σ={sig_slice:.2f} km/s)',
    fontsize=12
)
plt.tight_layout()
plt.savefig('results/eq_grid_DTM.png', dpi=150, bbox_inches='tight')

# ── Survey: total DTM vs nH at different T for this Z, sigma ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
DTM_total_slice = DTM_slice.sum(axis=-1)   # (nT, nNH)

ax = axes[0]
cm = plt.get_cmap('plasma', len(T_grid))
for i_T, T_val in enumerate(T_grid[::3]):
    i_T_idx = np.searchsorted(T_grid, T_val)
    ax.plot(log_nH_g, np.log10(np.clip(DTM_total_slice[i_T_idx, :], 1e-6, 1)),
            color=cm(i_T_idx/len(T_grid)), label=f'T={T_val:.0e} K')
ax.set_xlabel(r'$\log_{10}(n_H)$')
ax.set_ylabel(r'$\log_{10}$(total DTM)')
ax.set_title(f'Z={Z_slice:.2f} Z☉, σ={sig_slice:.2f} km/s')
ax.legend(fontsize=7, ncol=2)
ax.axhline(np.log10(np.average(DTM_sim[DTM_sim>0], weights=cell_mass[DTM_sim>0])),
           color='k', lw=2, ls='--', label='sim mean DTM')

ax = axes[1]   # same but vs Z at fixed (T, nH)
i_T_ref  = np.searchsorted(T_grid, 1e4)
i_nH_ref = np.searchsorted(nH_grid, 1.0)
DTM_vs_Z = rho_dust_4d[i_T_ref, i_nH_ref, :, i_sig_slice, :].sum(axis=-1)
rho_metal_Z = Z_grid_Zsun * ZSUN * (nH_grid[i_nH_ref] * MH * MU)
DTM_vs_Z /= rho_metal_Z
ax.plot(np.log10(Z_grid_Zsun), np.log10(np.clip(DTM_vs_Z, 1e-6, 1)),
        'o-', color='steelblue', label='Equilibrium')
ax.plot(np.log10(Z_grid_Zsun),
        np.log10([remy_ruyer_dtm(Z) for Z in Z_grid_Zsun]),
        'k--', label='Rémy-Ruyer+2014')
ax.set_xlabel(r'$\log_{10}(Z/Z_\odot)$')
ax.set_ylabel(r'$\log_{10}$(total DTM)')
ax.set_title(f'T={T_grid[i_T_ref]:.0e} K, nH={nH_grid[i_nH_ref]:.1f} cm⁻³, σ={sig_slice:.2f} km/s')
ax.legend()

plt.tight_layout()
plt.savefig('results/eq_DTM_vs_Z_and_nH.png', dpi=150, bbox_inches='tight')


## 9. Interpolate equilibrium onto simulation cells

In [ ]:
# ── Build 4D RegularGridInterpolator for each dust bin ────────────────────────
# Axes: (log10 T, log10 nH, log10 Z/Z_sun, log10 sigma)
log_T_ax  = np.log10(T_grid)
log_nH_ax = np.log10(nH_grid)
log_Z_ax  = np.log10(Z_grid_Zsun)
log_sig_ax = np.log10(sigma_grid)

# DTM grid: normalise by the metallicity and gas density at each (nH, Z) point
# rho_metal(nH, Z) = Z × Z_sun × nH × mH × mu  →  broadcast over (nT, nNH, nZ, nSig)
rho_gas_4d   = nH_grid[np.newaxis, :, np.newaxis, np.newaxis] * MH * MU  # (1,nNH,1,1)
rho_metal_4d = (Z_grid_Zsun[np.newaxis, np.newaxis, :, np.newaxis]
                * ZSUN * rho_gas_4d)                                       # (1,nNH,nZ,1)
DTM_4d = rho_dust_4d / rho_metal_4d[..., np.newaxis]  # (nT, nNH, nZ, nSig, 4)
DTM_4d = np.clip(DTM_4d, 0.0, 1.0)

interps_4d = [
    RegularGridInterpolator(
        (log_T_ax, log_nH_ax, log_Z_ax, log_sig_ax),
        DTM_4d[:, :, :, :, k],
        method='linear', bounds_error=False, fill_value=np.nan
    )
    for k in range(DTM_4d.shape[-1])
]

# ── Map equilibrium DTM to each simulation cell (use per-cell Z and sigma) ───
# Clip to grid bounds to avoid NaN from extrapolation
T_clipped   = np.clip(T_gas,   T_grid[0],        T_grid[-1])
nH_clipped  = np.clip(nH,      nH_grid[0],       nH_grid[-1])
Z_clipped   = np.clip(Z_metal / ZSUN, Z_grid_Zsun[0],  Z_grid_Zsun[-1])
sig_clipped = np.clip(sigma,   sigma_grid[0],    sigma_grid[-1])

# Cells with sigma == 0 (no turbulence field): use the lowest grid sigma
sig_clipped = np.where(sig_clipped > 0, sig_clipped, sigma_grid[0])

pts = np.column_stack([
    np.log10(T_clipped),
    np.log10(nH_clipped),
    np.log10(Z_clipped),
    np.log10(sig_clipped),
])

DTM_eq = np.column_stack([interp(pts) for interp in interps_4d])  # (ncells, 4)
DTM_eq = np.clip(DTM_eq, 0.0, None)

# Equilibrium dust densities (scale by per-cell metal mass density)
rho_eq          = DTM_eq * rho_metal[:, np.newaxis]   # (ncells, 4)
rho_cs_eq       = rho_eq[:, 0]
rho_cl_eq       = rho_eq[:, 1]
rho_ss_eq       = rho_eq[:, 2]
rho_sl_eq       = rho_eq[:, 3]
rho_dust_eq     = rho_eq.sum(axis=1)

out_of_bounds   = np.isnan(DTM_eq).any(axis=1)
mass_oob        = cell_mass[out_of_bounds].sum() / cell_mass.sum() * 100
print(f'Cells outside 4D grid bounds: {out_of_bounds.sum():,} ({mass_oob:.2f}% of total mass)')

with np.errstate(invalid='ignore', divide='ignore'):
    DTM_eq_total = np.where(rho_metal > 0, rho_dust_eq / rho_metal, 0.0)

good = ~out_of_bounds
print(f'Global DTM (simulation):  {(rho_dust    * vol).sum() / (rho_metal * vol).sum():.4f}')
print(f'Global DTM (equilibrium): {(rho_dust_eq[good] * vol[good]).sum() / (rho_metal[good] * vol[good]).sum():.4f}')

## 10. Comparison: T-nH phase diagrams

Side-by-side comparison of the simulation dust density vs. the equilibrium prediction on the T-nH plane.

In [ ]:
EQ_BINS = [rho_cs_eq, rho_cl_eq, rho_ss_eq, rho_sl_eq]

# Global colour limits based on simulation
ref_dust = np.concatenate([b[b > 0] for b in SIM_BINS])
vmin_common = np.log10(np.percentile(ref_dust, 5))
vmax_common = np.log10(np.percentile(ref_dust, 95))

fig_sim = phase_rho_dust(
    T_gas, nH, SIM_BINS, BIN_LABELS, vol,
    vmin=vmin_common, vmax=vmax_common,
    title='RAMSES simulation: dust mass density on T-nH plane'
)
plt.savefig('results/compare_sim_Tnplane.png', dpi=150, bbox_inches='tight')

fig_eq = phase_rho_dust(
    T_gas, nH, EQ_BINS, BIN_LABELS, vol,
    vmin=vmin_common, vmax=vmax_common,
    title='pyCALIMA equilibrium prediction on T-nH plane'
)
plt.savefig('results/compare_eq_Tnplane.png', dpi=150, bbox_inches='tight')


In [ ]:
# ── Ratio map: log₁₀(simulation / equilibrium) ───────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), sharey=True)
log_nH_c = np.log10(np.clip(nH,    1e-5, 1e4))
log_T_c  = np.log10(np.clip(T_gas, 1e0,  1e9))

for ax, rho_sim_bin, rho_eq_bin, label in zip(axes, SIM_BINS, EQ_BINS, BIN_LABELS):
    with np.errstate(invalid='ignore', divide='ignore'):
        log_ratio = np.log10((rho_sim_bin + 1e-40) / (rho_eq_bin + 1e-40))

    h_ratio, xe, ye = np.histogram2d(log_nH_c, log_T_c, bins=NBINS_2D,
                                      range=[NH_RANGE, T_RANGE],
                                      weights=log_ratio * vol)
    h_vol, _, _     = np.histogram2d(log_nH_c, log_T_c, bins=NBINS_2D,
                                      range=[NH_RANGE, T_RANGE],
                                      weights=vol)
    with np.errstate(invalid='ignore'):
        mean_ratio = np.where(h_vol > 0, h_ratio / h_vol, np.nan)

    im = ax.pcolormesh(xe, ye, mean_ratio.T, cmap='RdBu_r', vmin=-2, vmax=2)
    ax.set_xlabel(r'$\log_{10}(n_H\ [\mathrm{cm}^{-3}])$')
    ax.set_title(label, fontsize=10)
    plt.colorbar(im, ax=ax, label=r'$\log_{10}(\rho_\mathrm{sim}/\rho_\mathrm{eq})$')

axes[0].set_ylabel(r'$\log_{10}(T\ [\mathrm{K}])$')
fig.suptitle(
    'Simulation / Equilibrium ratio  (red = sim > eq, blue = sim < eq)',
    fontsize=12
)
plt.tight_layout()
plt.savefig('results/compare_ratio_Tnplane.png', dpi=150, bbox_inches='tight')


## 11. Comparison: global distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# ── Panel 1: total DTM histogram ──────────────────────────────────────────────
ax = axes[0, 0]
good = (DTM_sim > 0) & (DTM_eq_total > 0) & (~out_of_bounds)
bins_dtm = np.linspace(-6, 1, 100)
ax.hist(np.log10(DTM_sim[good]), bins=bins_dtm, weights=cell_mass[good],
        density=True, alpha=0.6, color='steelblue', label='Simulation')
ax.hist(np.log10(DTM_eq_total[good]), bins=bins_dtm, weights=cell_mass[good],
        density=True, alpha=0.6, color='darkorange', label='Equilibrium')
ax.set_xlabel(r'$\log_{10}$(DTM)')
ax.set_ylabel('Mass-weighted PDF')
ax.set_title('Total dust-to-metal ratio')
ax.legend()

# ── Panel 2: carb vs sil DTM ──────────────────────────────────────────────────
ax = axes[0, 1]
DTM_carb_sim = np.where(rho_metal > 0, (rho_cs + rho_cl) / rho_metal, 0)
DTM_sil_sim  = np.where(rho_metal > 0, (rho_ss + rho_sl) / rho_metal, 0)
DTM_carb_eq  = np.where(rho_metal > 0, (rho_cs_eq + rho_cl_eq) / rho_metal, 0)
DTM_sil_eq   = np.where(rho_metal > 0, (rho_ss_eq + rho_sl_eq) / rho_metal, 0)

color_T = np.log10(np.clip(T_gas[good], 1e1, 1e8))
sc = ax.scatter(np.log10(DTM_carb_sim[good] + 1e-7),
                np.log10(DTM_sil_sim[good]  + 1e-7),
                c=color_T, cmap='RdYlBu_r', s=0.1, alpha=0.3, label='Sim')
ax.scatter(np.log10(DTM_carb_eq[good] + 1e-7),
           np.log10(DTM_sil_eq[good]  + 1e-7),
           c=color_T, cmap='RdYlBu_r', s=0.1, alpha=0.3, marker='s', label='Eq')
plt.colorbar(sc, ax=ax, label=r'$\log_{10}(T)$')
ax.set_xlabel(r'$\log_{10}$(DTM carbonaceous)')
ax.set_ylabel(r'$\log_{10}$(DTM silicate)')
ax.set_title('Carbonaceous vs. silicate (circles=sim, sq=eq)')

# ── Panel 3: small/large ratio per composition ────────────────────────────────
ax = axes[1, 0]
pos = good & (rho_cl > 0) & (rho_sl > 0) & (rho_cl_eq > 0) & (rho_sl_eq > 0)
ratio_carb_sim = np.log10((rho_cs[pos] + 1e-40) / (rho_cl[pos] + 1e-40))
ratio_sil_sim  = np.log10((rho_ss[pos] + 1e-40) / (rho_sl[pos] + 1e-40))
ratio_carb_eq  = np.log10((rho_cs_eq[pos] + 1e-40) / (rho_cl_eq[pos] + 1e-40))
ratio_sil_eq   = np.log10((rho_ss_eq[pos] + 1e-40) / (rho_sl_eq[pos] + 1e-40))

bins_r = np.linspace(-4, 4, 80)
ax.hist(ratio_carb_sim, bins=bins_r, weights=cell_mass[pos], density=True,
        alpha=0.5, color='steelblue', label='Carb sim')
ax.hist(ratio_carb_eq,  bins=bins_r, weights=cell_mass[pos], density=True,
        alpha=0.5, color='steelblue', histtype='step', lw=2, label='Carb eq')
ax.hist(ratio_sil_sim,  bins=bins_r, weights=cell_mass[pos], density=True,
        alpha=0.5, color='darkorange', label='Sil sim')
ax.hist(ratio_sil_eq,   bins=bins_r, weights=cell_mass[pos], density=True,
        alpha=0.5, color='darkorange', histtype='step', lw=2, label='Sil eq')
ax.set_xlabel(r'$\log_{10}(M_\mathrm{small}/M_\mathrm{large})$')
ax.set_ylabel('Mass-weighted PDF')
ax.set_title('Small-to-large mass ratio (filled=sim, step=eq)')
ax.legend(fontsize=8)

# ── Panel 4: 1-to-1 scatter (total dust) ─────────────────────────────────────
ax = axes[1, 1]
pos2 = good & (rho_dust > 0) & (rho_dust_eq > 0)
sc2 = ax.scatter(np.log10(rho_dust_eq[pos2]), np.log10(rho_dust[pos2]),
                 c=np.log10(T_gas[pos2]), cmap='RdYlBu_r',
                 s=0.3, alpha=0.3, rasterized=True)
lims = (-37, -25)
ax.plot(lims, lims, 'k--', lw=1, label='1:1')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel(r'$\log_{10}(\rho_\mathrm{dust,eq})$ [g/cm³]')
ax.set_ylabel(r'$\log_{10}(\rho_\mathrm{dust,sim})$ [g/cm³]')
ax.set_title('Simulation vs. equilibrium (total dust)')
plt.colorbar(sc2, ax=ax, label=r'$\log_{10}(T)$')
ax.legend()

plt.tight_layout()
plt.savefig('results/compare_global.png', dpi=150, bbox_inches='tight')


In [ ]:
# ── Per-bin 1-to-1 scatter ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
lims = (-38, -25)

for ax, rho_sim_bin, rho_eq_bin, label in zip(axes, SIM_BINS, EQ_BINS, BIN_LABELS):
    pos_b = good & (rho_sim_bin > 0) & (rho_eq_bin > 0)
    sc = ax.scatter(
        np.log10(rho_eq_bin[pos_b]), np.log10(rho_sim_bin[pos_b]),
        c=np.log10(T_gas[pos_b]), cmap='RdYlBu_r', vmin=1, vmax=7,
        s=0.3, alpha=0.3, rasterized=True
    )
    ax.plot(lims, lims, 'k--', lw=1.5)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel(r'$\log_{10}(\rho_\mathrm{eq})$ [g/cm³]')
    ax.set_ylabel(r'$\log_{10}(\rho_\mathrm{sim})$ [g/cm³]')
    ax.set_title(label, fontsize=10)
    plt.colorbar(sc, ax=ax, label=r'$\log_{10}(T)$')
    # Print Pearson r in log space
    r = np.corrcoef(np.log10(rho_eq_bin[pos_b]), np.log10(rho_sim_bin[pos_b]))[0, 1]
    ax.text(0.05, 0.92, f'r = {r:.3f}', transform=ax.transAxes, fontsize=10,
            color='black', bbox=dict(boxstyle='round', fc='wheat', alpha=0.7))

fig.suptitle('Per-bin: simulation vs. equilibrium dust mass density', fontsize=12)
plt.tight_layout()
plt.savefig('results/compare_per_bin_scatter.png', dpi=150, bbox_inches='tight')


## 10. Summary and interpretation

### What the comparison shows

- **Agreement regime**: In the warm neutral/molecular medium (T ~ 100–8000 K, nH ~ 1–100 cm⁻³) the simulation should approach equilibrium values if the run time is long enough relative to the accretion timescale τ_acc ~ 1/(nH × k_acc).

- **Simulation below equilibrium** (blue in ratio maps): typically in dense, cold gas (T < 300 K, nH > 100 cm⁻³) where dust accretion is fast but the simulation DTMini = 10⁻³ starting point may not have had time to reach equilibrium, *or* the equilibrium predicts very high DTM that has not been reached.

- **Simulation above equilibrium** (red in ratio maps): typically in hot, diffuse gas (T > 10⁵ K, nH < 0.1 cm⁻³) where the equilibrium predicits near-zero dust (thermal sputtering destroys it), but the simulation retains some dust that was advected from denser regions before being sputtered.

### Caveats
1. **G0 = 1 (assumed)**: The UV field varies across the galaxy. Cells near star-forming regions could have G0 ≫ 1, which would reduce the equilibrium DTM (more photoelectric heating → less accretion).
2. **Fixed σ**: The turbulence parameter is set to the mass-weighted mean; individual cells with much higher/lower σ will have different coagulation/shattering equilibria.
3. **Fixed 0.1 Z☉ budget**: Per-cell element abundances vary; the equilibrium is scaled by the cell's *gas-phase* metallicity, which underestimates the total budget by the amount locked in dust.
4. **No PAH bins in RAMSES**: The 4 dust bins in RAMSES have no PAH equivalent; pyCALIMA's PAH physics is not exercised here.

### Next steps
- Extend the grid to include G0 as a third axis (estimated from local SFR density)
- Use per-cell total element budget (gas + dust) for a more accurate equilibrium constraint
- Add a time-evolution comparison: run pyCALIMA RK4 from the simulation's initial DTM and check if it converges to the simulation values